In [1]:
import os
import re
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Model Training

In [2]:
log_folder = '/scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/slurms/'
file_names = ['5033695_1','4854100_2','4854310_4']

In [3]:
sub_list = ['sub-01', 'sub-02', 'sub-04']
logs = {}
for i, sub in enumerate(sub_list):
    print(file_names[i])
    print(sub)
    file_path = os.path.join(log_folder, f'{file_names[i]}.err')
    with open(file_path, 'r') as f:
        logs[sub] = f.read()

5033695_1
sub-01
4854100_2
sub-02
4854310_4
sub-04


In [4]:
# PixCorr (pixel correlation), SSIM (structural similarity in-
# dex metric; Zhou Wang et al., 2004), EfficientNet-B1 (“Eff”;
# Tan & Le, 2020) and SwAV-ResNet50 (“SwAV”; Caron
# et al., 2021) compute the average correlation distance be-
# tween the ground truth and reconstructed image. AlexNet
# (layers 2 and 5) (Krizhevsky et al., 2012), Inception-v3 (last
# pooling layer) (Szegedy et al., 2016), and CLIP (last layer
# of ViT-L/14) (Radford et al., 2021) use two-way compar-
# isons based on extracted features from the specified layer
# (chance=50%). For each test image, we compute the Pear-
# son’s correlation of each ground truth image with each re-
# construction in the test set; two-way accuracy refers to the
# number of times the correct ground-truth reconstruction pair
# is more correlated than a mismatched pair, averaged over all
# possible pairwise comparisons and then over all test images
# to produce a single score.


#test/loss
#test/loss_clip_total
#test/loss_prior
#test/num_steps
#test/recon_cossim
#test/recon_mse
#test/test_bwd_pct_correct
#test/test_fwd_pct_correct
#train/bwd_pct_correct=0.0362
#train/fwd_pct_correct=0.0458, 

#train/loss
#train/loss_clip_total
#train/loss_prior

#train/lr
#train/num_steps
#train/recon_cossim
#train/recon_mse=0.346

metrics = ['train/loss', 'train/num_steps', 'train/fwd_pct_correct','train/bwd_pct_correct',
           'test/loss', 'test/num_steps', 'test/test_fwd_pct_correct','test/test_bwd_pct_correct',]

In [5]:
def extract_num(keyword, string):
    # Pattern explanation:
    # (?<=" + keyword + "=) is a positive lookbehind assertion, matching the position after the keyword and "=".
    # \\d+ matches one or more digits.
    pattern = r"(?<=" + keyword + r"=)\d+\.?\d*"
    
    # re.findall returns all non-overlapping matches of the pattern as a list of strings.
    numbers_str_list = re.findall(pattern, string)

    # Convert the list of string numbers to a list of integers
    numbers_list = [float(num) for num in numbers_str_list]

    print(f"{keyword} found {len(numbers_list)}.")

    return numbers_list

In [6]:
num_dict = {}

for sub in sub_list:
    num_dict[sub] = {}
    for i, metric in enumerate(metrics):
        num_dict[sub][metric] = extract_num(metric, logs[sub])

train/loss found 301.
train/num_steps found 301.
train/fwd_pct_correct found 301.
train/bwd_pct_correct found 301.
test/loss found 301.
test/num_steps found 301.
test/test_fwd_pct_correct found 301.
test/test_bwd_pct_correct found 301.
train/loss found 301.
train/num_steps found 301.
train/fwd_pct_correct found 301.
train/bwd_pct_correct found 301.
test/loss found 301.
test/num_steps found 301.
test/test_fwd_pct_correct found 301.
test/test_bwd_pct_correct found 301.
train/loss found 301.
train/num_steps found 301.
train/fwd_pct_correct found 301.
train/bwd_pct_correct found 301.
test/loss found 301.
test/num_steps found 301.
test/test_fwd_pct_correct found 301.
test/test_bwd_pct_correct found 301.


In [7]:
df = pd.DataFrame(columns=metrics+['sub'])

for sub in sub_list:
    sub_df = pd.DataFrame(num_dict[sub])
    sub_df['sub'] = sub
    
    df = pd.concat([df, sub_df], ignore_index=True)

/tmp/ipykernel_2260465/897763415.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, sub_df], ignore_index=True)


In [8]:
df.to_csv("bixby_data/loss_acc.csv")

## Model Eval

In [2]:
import pickle
from tqdm import tqdm
from collections import defaultdict
from scipy.stats import zscore

sub = 'sub-02'

In [3]:
def find_repeated_strings(string_list):
    """
    Finds all repeated strings in a list and returns a dictionary 
    with the string as the key and a list of its indices as the value.
    Uses a set to track seen items efficiently.
    """
    # Set to quickly track which items have appeared once already
    seen_once = set()
    # Dictionary to store only the indices of items that repeat
    repeated_strings_dict = {}

    for index, string_val in enumerate(string_list):
        if string_val in repeated_strings_dict:
            # If already in the 'repeated_strings_dict', just append the new index
            repeated_strings_dict[string_val].append(index)
        elif string_val in seen_once:
            # First time seeing a repeat: move from 'seen_once' to 'repeated_strings_dict'
            repeated_strings_dict[string_val] = [string_list.index(string_val), index]
        else:
            # First time seeing the item overall
            seen_once.add(string_val)
            
    return repeated_strings_dict

def locate_repeat_index_per_run(sub_dict, unique_idx):
    
    runs = sub_dict['run']
    unique_runs = list(set(runs))
    trials = ["_".join(trial.split('_')[1:-1]) for trial in sub_dict['trial']]
    repeated_trial = find_repeated_strings(trials)

    # structure output idx dictionary
    unique_runs.sort()
    default_value = {}
    sorted_vox = dict.fromkeys(unique_runs, default_value)
    for k in sorted_vox.keys():
        sorted_vox[k] = defaultdict(list)
        
    for trial in test_images:
        idx_list = repeated_trial[trial]
        for i in idx_list:
            curr_run = runs[i]
            sorted_vox[curr_run][trial].append(i)
    
    return repeated_trial, sorted_vox

def average_repeats_snap(vox, repeated_trial, unique_images):
        
    sorted_vox = np.zeros((len(unique_images), vox.shape[1]))
    assert len(repeated_trial.keys()) == len(unique_images)
    
    # Average repeated MST images
    for i, img in enumerate(unique_images):

        if img in repeated_trial.keys(): # deal with repeated images
            # average all repeats across sessions
            curr_trial_vox = np.mean(vox[repeated_trial[img]], axis=0)
            
        else: # error handeling
            print(f"{img} is not in the list")
            break
        
        sorted_vox[i, :] = curr_trial_vox
    
    return sorted_vox

In [4]:
dic = {}

test_images = [f'A_{i}' for i in range(1,19)] + [f'B_{i}' for i in range(1,19)]

data_folder = '/scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/'

sessions = ['study', 'test', 'snap']

folder_path = os.path.join(data_folder, 'afni')

with open(f'{folder_path}/{sub}_roi_vox_all_sessions.pkl', 'rb') as file:
    dic[sub] = pickle.load(file)
    
union_mask = dic[sub]['union_mask']
print('NSD mask size:', union_mask.shape)
print('union mask size:', sum(union_mask))
for ses in sessions:
    dic[sub][ses]['roi'] = dic[sub][ses]['roi'][:, union_mask]
    # z-score each session
    dic[sub][ses]['roi'] = np.nan_to_num(zscore(dic[sub][ses]['roi'], axis=0))
    s = dic[sub][ses]['roi'].shape
    print(f'{ses}: {s}')

NSD mask size: (17175,)
union mask size: 3643
study: (216, 3643)
test: (216, 3643)
snap: (432, 3643)


In [5]:
idx_dict = {}

idx_dict[sub] = {}

for session in sessions:

    sub_dict = dic[sub][session]

    _, per_run_repeat_idx = locate_repeat_index_per_run(sub_dict, test_images)
    
    idx_dict[sub][session] = per_run_repeat_idx

In [39]:
dic[sub][ses]['roi']

3643

In [63]:
run_dict = {}

for run in idx_dict[sub][session].keys():
    print(run)
    run_dict[run] = {}
        
    for img in test_images:        
        img_vox = []
        for session in sessions:
            vox = dic[sub][session]['roi']
            idx = idx_dict[sub][session][run][img]
            for i in idx:
                img_vox.append(vox[i,:])
                
        #run_dict[img] = np.array(img_vox)
        run_dict[run][img] = np.mean(np.array(img_vox), axis=0)

run-01
run-02
run-03
run-04
run-05
run-06


In [65]:
test_img = None
img_path = f'/scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/afni/loaded_test_imgs.pkl'

if os.path.exists(img_path):
    with open(img_path, 'rb') as file:
        test_img = pickle.load(file)
        print('Loading image saved at: ', file)
else:
    for img in test_images:

        root_dir = os.path.join(data_folder, 'stimuli', 'scenes')
        img_list = img.split('_')[0]
        img_id = int(img.split('_')[1])
        
        image_file = f'{root_dir}/list{img_list}/{img_id:02d}.png'

        if image_file and not os.path.exists(image_file):
            print('Cannot find the image at this path',image_file)
            break

        im = imageio.imread(image_file)
        im = torch.Tensor(im / 255).permute(2,0,1)
        im = resize_transform(im.unsqueeze(0))

        if test_img is None:
            test_img = im
        else:
            test_img = torch.vstack((test_img, im))

    
    print(folder_path)
    with open(img_path, 'wb') as file:
        pickle.dump(test_img, file)
        print('image saved at: ', file)
        
print("testing images", test_img.shape)

Loading image saved at:  <_io.BufferedReader name='/scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/afni/loaded_test_imgs.pkl'>
testing images torch.Size([36, 3, 224, 224])


### Loading pre-trained subject model

In [66]:
outdir_folder = '/scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/'
folder_names = {'sub-02':'sub-02_output_1771391206.9882252', 'sub-04':'sub-04_output_1771391422.1811972'}

outdir = os.path.join(outdir_folder, folder_names[sub])

In [67]:
outdir

'/scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/sub-02_output_1771391206.9882252'

In [68]:
def load_ckpt(tag,load_lr=True,load_optimizer=True,load_epoch=True,strict=True,outdir=outdir,multisubj_loading=False): 
    print(f"\n---loading {outdir}/{tag}.pth ckpt---\n")
    checkpoint = torch.load(outdir+'/last.pth', map_location='cpu')
    state_dict = checkpoint['model_state_dict']
    if multisubj_loading: # remove incompatible ridge layer that will otherwise error
        state_dict.pop('ridge.linears.0.weight',None)
    model.load_state_dict(state_dict, strict=strict)
    if load_epoch:
        globals()["epoch"] = checkpoint['epoch']
        print("Epoch",epoch)
    if load_optimizer:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if load_lr:
        lr_scheduler.load_state_dict(checkpoint['lr_scheduler'])
    del checkpoint

In [69]:
import torch
import torch.nn as nn
### Multi-GPU config ###
from accelerate import Accelerator, DeepSpeedPlugin

local_rank = os.getenv('RANK')
if local_rank is None: 
    local_rank = 0
else:
    local_rank = int(local_rank)
print("LOCAL RANK ", local_rank)  

data_type = torch.float32 # change depending on your mixed_precision

accelerator = Accelerator(split_batches=False)

test_vox = torch.Tensor(dic[sub][session]['roi'])
test_img = torch.Tensor(test_img)

device = accelerator.device

LOCAL RANK  0


In [70]:
from models import PriorNetwork, BrainDiffusionPrior
import utils

## USING OpenCLIP ViT-bigG ###
sys.path.append('generative_models/')
import sgm
from generative_models.sgm.modules.encoders.modules import FrozenOpenCLIPImageEmbedder
# from generative_models.sgm.models.diffusion import DiffusionEngine
# from omegaconf import OmegaConf

torch.cuda.empty_cache()

data_type = torch.float32

num_voxels_list = [dic[sub]['study']['roi'].shape[-1]]
n_blocks=4
hidden_dim=1024
use_prior=False
clip_scale=1.

try:
    print(clip_img_embedder)
except:
    clip_img_embedder = FrozenOpenCLIPImageEmbedder(
        arch="ViT-bigG-14",
        version="laion2b_s39b_b160k",
        output_tokens=True,
        only_tokens=True,
    )
    clip_img_embedder.to(device)
clip_img_embedder.model.visual.set_grad_checkpointing(True)

clip_seq_dim = 256
clip_emb_dim = 1664

model = utils.prepare_model_and_training(
    num_voxels_list=num_voxels_list,
    n_blocks=n_blocks,
    hidden_dim=hidden_dim,
    clip_emb_dim=clip_emb_dim,
    clip_seq_dim=clip_seq_dim,
    use_prior=use_prior,
    clip_scale=clip_scale
)

MindEyeModule()
param counts:
3,731,456 total
3,731,456 trainable
param counts:
3,731,456 total
3,731,456 trainable
param counts:
453,360,280 total
453,360,280 trainable
param counts:
457,091,736 total
457,091,736 trainable


In [71]:
load_ckpt("last",
          outdir=outdir,load_lr=False,load_optimizer=False,load_epoch=False,strict=False,multisubj_loading=True)
model.to(device)


---loading /scratch/gpfs/KNORMAN/wanjia/mindeye_testing/real_time_mindEye2/bixby_data/sub-02_output_1771391206.9882252/last.pth ckpt---



MindEyeModule(
  (ridge): RidgeRegression(
    (linears): ModuleList(
      (0): Linear(in_features=3643, out_features=1024, bias=True)
    )
  )
  (backbone): BrainNetwork(
    (mixer_blocks1): ModuleList(
      (0-3): 4 x Sequential(
        (0): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (1): Sequential(
          (0): Linear(in_features=1024, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.15, inplace=False)
          (3): Linear(in_features=1024, out_features=1024, bias=True)
        )
      )
    )
    (mixer_blocks2): ModuleList(
      (0-3): 4 x Sequential(
        (0): LayerNorm((1,), eps=1e-05, elementwise_affine=True)
        (1): Sequential(
          (0): Linear(in_features=1, out_features=1, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.15, inplace=False)
          (3): Linear(in_features=1, out_features=1, bias=True)
        )
      )
    )
    (backbone_linear): Linear(i

In [109]:
def evaluate_mst_pairs(vox_idx, vox, img_idx, images):

    score = 0
    total = 0
    
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=data_type):
        for i, pair in enumerate(vox_idx):
            voxel = vox[pair[0]].to(device)[None]
            voxel = torch.Tensor(voxel).unsqueeze(1).to(device)
            
            imageA = images[img_idx[i][0]].to(device)[None]
            imageB = images[img_idx[i][1]].to(device)[None]
            
            clip_targetA = clip_img_embedder(imageA.float())
            clip_targetB = clip_img_embedder(imageB.float())
            
            voxel_ridge = model.ridge(voxel,0)
            backbone, clip_voxels, _ = model.backbone(voxel_ridge)
            
            clip_voxels_norm = nn.functional.normalize(clip_voxels.flatten(1), dim=-1)
            clip_targetA_norm = nn.functional.normalize(clip_targetA.flatten(1), dim=-1)
            clip_targetB_norm = nn.functional.normalize(clip_targetB.flatten(1), dim=-1)
            
            voxel1 = clip_voxels_norm
            
            
            if utils.batchwise_cosine_similarity(clip_voxels_norm, clip_targetA_norm) > utils.batchwise_cosine_similarity(clip_voxels_norm, clip_targetB_norm):
                score += 1
            total += 1
            
            voxel = vox[pair[1]].to(device)[None]
            voxel = torch.Tensor(voxel).unsqueeze(1).to(device)
            
            voxel_ridge = model.ridge(voxel,0)
            backbone, clip_voxels, _ = model.backbone(voxel_ridge)
            clip_voxels_norm = nn.functional.normalize(clip_voxels.flatten(1), dim=-1)
            
            if utils.batchwise_cosine_similarity(clip_voxels_norm, clip_targetB_norm) > utils.batchwise_cosine_similarity(clip_voxels_norm, clip_targetA_norm):
                score += 1
            total += 1
            
            print('images similarity:', utils.batchwise_cosine_similarity(imageA, imageB))
            print('voxels similarity:', utils.batchwise_cosine_similarity(voxel1, voxel))
            print('image/voxel similarity:', utils.batchwise_cosine_similarity(voxel1, clip_targetA_norm))
            print('comp_image/voxel similarity:',utils.batchwise_cosine_similarity(voxel1, clip_targetB_norm))

        print('score:', score)
        print('total:', total)
            
    return score/total

In [110]:
scores = {}

for run in run_dict.keys():
    
    scores[run] = []
    
    vox_keys = list(run_dict[run].keys())
    vox_matrix= torch.Tensor(np.array(list(run_dict['run-01'].values())))
    
    mst_pairs_vox_index = []
    mst_pairs_img_index = []

    for i in range(1,19):
        
        a = f'A_{i}'
        b = f'B_{i}'

        mst_pairs_vox_index.append([vox_keys.index(a), vox_keys.index(b)])
        mst_pairs_img_index.append([test_images.index(a), test_images.index(b)])

    print(mst_pairs_vox_index)
    print(mst_pairs_img_index)

    assert len(mst_pairs_vox_index) == len(mst_pairs_img_index)
    
    afc = evaluate_mst_pairs(mst_pairs_vox_index, vox_matrix, mst_pairs_img_index, test_img)
    scores[run].append(afc)
    
    break
        

[[0, 18], [1, 19], [2, 20], [3, 21], [4, 22], [5, 23], [6, 24], [7, 25], [8, 26], [9, 27], [10, 28], [11, 29], [12, 30], [13, 31], [14, 32], [15, 33], [16, 34], [17, 35]]
[[0, 18], [1, 19], [2, 20], [3, 21], [4, 22], [5, 23], [6, 24], [7, 25], [8, 26], [9, 27], [10, 28], [11, 29], [12, 30], [13, 31], [14, 32], [15, 33], [16, 34], [17, 35]]
images similarity: tensor([[0.9230]], device='cuda:0')
image/voxel similarity: tensor([[0.0879]], device='cuda:0')
comp_image/voxel similarity: tensor([[0.0872]], device='cuda:0')
images similarity: tensor([[0.9292]], device='cuda:0')
image/voxel similarity: tensor([[0.0928]], device='cuda:0')
comp_image/voxel similarity: tensor([[0.0883]], device='cuda:0')
images similarity: tensor([[0.9147]], device='cuda:0')
image/voxel similarity: tensor([[0.0892]], device='cuda:0')
comp_image/voxel similarity: tensor([[0.0857]], device='cuda:0')
images similarity: tensor([[0.9088]], device='cuda:0')
image/voxel similarity: tensor([[0.0875]], device='cuda:0')
com

In [99]:
scores

{'run-01': [0.5277777777777778],
 'run-02': [0.4722222222222222],
 'run-03': [0.4722222222222222],
 'run-04': [0.5],
 'run-05': [0.4166666666666667],
 'run-06': [0.3888888888888889]}

In [89]:
scores = {}

for run in run_dict.keys():
    
    vox_keys = list(run_dict[run].keys())
    vox_matrix= torch.Tensor(np.array(list(run_dict['run-01'].values())))
    
    mst_pairs_vox_index = []
    mst_pairs_img_index = []

    for i in range(1,19):
        
        a = f'A_{i}'
        b = f'B_{i}'

        mst_pairs_vox_index.append([vox_keys.index(a), vox_keys.index(b)])
        mst_pairs_img_index.append([test_images.index(a), test_images.index(b)])

    print(mst_pairs_vox_index)
    print(mst_pairs_img_index)

    assert len(mst_pairs_vox_index) == len(mst_pairs_img_index)
    
    afc = evaluate_mst_pairs(mst_pairs_vox_index, vox_matrix, mst_pairs_img_index, test_img)
    scores[run].append(afc)
    
    break
        

[[0, 18], [1, 19], [2, 20], [3, 21], [4, 22], [5, 23], [6, 24], [7, 25], [8, 26], [9, 27], [10, 28], [11, 29], [12, 30], [13, 31], [14, 32], [15, 33], [16, 34], [17, 35]]
[[0, 18], [1, 19], [2, 20], [3, 21], [4, 22], [5, 23], [6, 24], [7, 25], [8, 26], [9, 27], [10, 28], [11, 29], [12, 30], [13, 31], [14, 32], [15, 33], [16, 34], [17, 35]]


AttributeError: 'numpy.ndarray' object has no attribute 'to'

In [ ]:
run_dict

In [55]:
scores = {}

for session in idx_dict[sub]:
    print(session)
    scores[session] = []
    
    if session == 'snap':
        input_i = [0, 1]
    else:
        input_i = [0]
    
    for run in idx_dict[sub][session]:
        for idx in input_i:
            
            curr_run = idx_dict[sub][session][run]
            print(run)

            mst_pairs_vox_index = []
            mst_pairs_img_index = []

            for i in range(1,19):
                a = f'A_{i}'
                b = f'B_{i}'

                mst_pairs_vox_index.append([curr_run[a][idx], curr_run[b][idx]])
                mst_pairs_img_index.append([test_images.index(a), test_images.index(b)])

            print(mst_pairs_vox_index)
            print(mst_pairs_img_index)

            assert len(mst_pairs_vox_index) == len(mst_pairs_img_index)

            afc = evaluate_mst_pairs(mst_pairs_vox_index, test_vox, mst_pairs_img_index, test_img)
            scores[session].append(afc)

study
run-01
[[9, 27], [10, 28], [11, 29], [12, 30], [13, 31], [14, 32], [15, 33], [16, 34], [17, 35], [0, 18], [1, 19], [2, 20], [3, 21], [4, 22], [5, 23], [6, 24], [7, 25], [8, 26]]
[[0, 18], [1, 19], [2, 20], [3, 21], [4, 22], [5, 23], [6, 24], [7, 25], [8, 26], [9, 27], [10, 28], [11, 29], [12, 30], [13, 31], [14, 32], [15, 33], [16, 34], [17, 35]]


/home/wg7536/.conda/envs/rt_mindEye2/lib/python3.11/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/wg7536/.conda/envs/rt_mindEye2/lib/python3.11/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


run-02
[[45, 63], [46, 64], [47, 65], [48, 66], [49, 67], [50, 68], [51, 69], [52, 70], [53, 71], [36, 54], [37, 55], [38, 56], [39, 57], [40, 58], [41, 59], [42, 60], [43, 61], [44, 62]]
[[0, 18], [1, 19], [2, 20], [3, 21], [4, 22], [5, 23], [6, 24], [7, 25], [8, 26], [9, 27], [10, 28], [11, 29], [12, 30], [13, 31], [14, 32], [15, 33], [16, 34], [17, 35]]
run-03
[[81, 99], [82, 100], [83, 101], [84, 102], [85, 103], [86, 104], [87, 105], [88, 106], [89, 107], [72, 90], [73, 91], [74, 92], [75, 93], [76, 94], [77, 95], [78, 96], [79, 97], [80, 98]]
[[0, 18], [1, 19], [2, 20], [3, 21], [4, 22], [5, 23], [6, 24], [7, 25], [8, 26], [9, 27], [10, 28], [11, 29], [12, 30], [13, 31], [14, 32], [15, 33], [16, 34], [17, 35]]
run-04
[[117, 135], [118, 136], [119, 137], [120, 138], [121, 139], [122, 140], [123, 141], [124, 142], [125, 143], [108, 126], [109, 127], [110, 128], [111, 129], [112, 130], [113, 131], [114, 132], [115, 133], [116, 134]]
[[0, 18], [1, 19], [2, 20], [3, 21], [4, 22], [5, 

In [56]:
scores # sub-04

{'study': [0.5833333333333334,
  0.4166666666666667,
  0.5833333333333334,
  0.5555555555555556,
  0.5833333333333334,
  0.5555555555555556],
 'test': [0.5833333333333334,
  0.3888888888888889,
  0.5833333333333334,
  0.5555555555555556,
  0.6111111111111112,
  0.5555555555555556],
 'snap': [0.3888888888888889,
  0.6111111111111112,
  0.4722222222222222,
  0.5277777777777778,
  0.4722222222222222,
  0.5555555555555556,
  0.5277777777777778,
  0.5,
  0.4722222222222222,
  0.4444444444444444,
  0.6111111111111112,
  0.6666666666666666]}

In [25]:
scores # sub-02

{'study': [0.5,
  0.3888888888888889,
  0.5277777777777778,
  0.5833333333333334,
  0.5,
  0.5555555555555556],
 'test': [0.4722222222222222,
  0.4166666666666667,
  0.5,
  0.6111111111111112,
  0.4444444444444444,
  0.5277777777777778],
 'snap': [0.5277777777777778,
  0.5,
  0.4722222222222222,
  0.5277777777777778,
  0.6111111111111112,
  0.5277777777777778,
  0.5555555555555556,
  0.5277777777777778,
  0.5277777777777778,
  0.75,
  0.5,
  0.3611111111111111]}

In [57]:
scores['snap']

[0.3888888888888889,
 0.6111111111111112,
 0.4722222222222222,
 0.5277777777777778,
 0.4722222222222222,
 0.5555555555555556,
 0.5277777777777778,
 0.5,
 0.4722222222222222,
 0.4444444444444444,
 0.6111111111111112,
 0.6666666666666666]

In [61]:
scores['snap1'] = scores['snap'][::2]
# Elements at odd indices (1, 3, 5, ...)
# Start at index 1, go to the end, step by 2
scores['snap2'] = scores['snap'][1::2]

In [64]:
import pandas as pd

snap = scores.pop('snap')
df = pd.DataFrame(scores)
df['sub'] = sub

print(snap)

[0.3888888888888889, 0.6111111111111112, 0.4722222222222222, 0.5277777777777778, 0.4722222222222222, 0.5555555555555556, 0.5277777777777778, 0.5, 0.4722222222222222, 0.4444444444444444, 0.6111111111111112, 0.6666666666666666]


In [70]:
pd.concat([df2, df]).to_csv('bixby_data/2afc.csv')

In [37]:
df.to_csv('bixby_data'

,study,test,snap1,snap2,sub
0,0.500000,0.472222,0.527778,0.500000,sub-02
1,0.388889,0.416667,0.472222,0.527778,sub-02
2,0.527778,0.500000,0.611111,0.527778,sub-02
3,0.583333,0.611111,0.555556,0.527778,sub-02
4,0.500000,0.444444,0.527778,0.750000,sub-02
5,0.555556,0.527778,0.500000,0.361111,sub-02


In [34]:
snap

[0.5277777777777778,
 0.5,
 0.4722222222222222,
 0.5277777777777778,
 0.6111111111111112,
 0.5277777777777778,
 0.5555555555555556,
 0.5277777777777778,
 0.5277777777777778,
 0.75,
 0.5,
 0.3611111111111111]

In [62]:
import copy

df2 = copy.deepcopy(df)

In [63]:
df2

,study,test,snap1,snap2,sub
0,0.500000,0.472222,0.527778,0.500000,sub-02
1,0.388889,0.416667,0.472222,0.527778,sub-02
2,0.527778,0.500000,0.611111,0.527778,sub-02
3,0.583333,0.611111,0.555556,0.527778,sub-02
4,0.500000,0.444444,0.527778,0.750000,sub-02
5,0.555556,0.527778,0.500000,0.361111,sub-02
